#Criar estrutura de dados no Unity Catalog

In [0]:
%run ./config

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS `{CATALOG}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")
spark.sql(f""" CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.`{VOLUME}` COMMENT 'Arquivos brutos coletados das APIs do MVP' """)

BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

print(f"Catálogo: {CATALOG}")
print(f"Schema: {SCHEMA}")
print(f"Volume: {BASE_PATH}")

In [0]:
df_validation = (spark.sql(f"""SELECT 'clinical_trials' AS fonte,
                                       COUNT(*) AS registros,
                                       COUNT(DISTINCT nct_id) AS registros_distintos
                                 FROM {CATALOG}.{SCHEMA}.brz_clinical_trials

                                UNION ALL

                               SELECT 'world_bank' AS fonte,
                                      COUNT(*) AS registros,
                                      COUNT(DISTINCT CONCAT(country_iso3, '-', year))
                                    FROM {CATALOG}.{SCHEMA}.brz_population"""))
display(df_validation)

In [0]:
tables = ["slv_studies","slv_locations","slv_locations_iso3","slv_interventions","slv_conditions","slv_population"]

for table_name in tables:
    count = spark.table(f"{CATALOG}.{SCHEMA}.{table_name}").count()
    print(f"{table_name}: {count:,} registros")

In [0]:
df_validation = spark.sql(f""" SELECT country_name,
                                      COUNT(DISTINCT nct_id) AS studies
                                 FROM {CATALOG}.{SCHEMA}.slv_locations_iso3
                                WHERE country_iso3 IS NULL
                                GROUP BY country_name
                                ORDER BY studies DESC""")
display(df_validation)


In [0]:
comments = {"slv_studies": "Estudos clínicos sobre câncer de mama normalizados.",
          "slv_locations": "Localizações geográficas dos estudos clínicos.",
     "slv_locations_iso3": "Localizações geográficas dos estudos clínicos mapeado para iso-3.",
      "slv_interventions": "Intervenções e tratamentos pesquisados.",
         "slv_conditions": "Condições médicas associadas aos estudos.",
         "slv_population": "População anual por país obtida do Banco Mundial."}

for table_name, comment in comments.items():
    spark.sql(f"""COMMENT ON TABLE `{CATALOG}`.`{SCHEMA}`.`{table_name}` IS '{comment}'""")